### Milvus操作

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

In [ ]:
dbs = client.list_databases()
db_name = "rag_demo"
if db_name not in dbs:
    client.create_database(db_name)
dbs = client.list_databases()
for db in dbs:
    print(db)
# client.drop_database(db_name)

In [ ]:
# 创建 collection
client.use_database(db_name)
collection_name = "docs"
client.create_collection(
    collection_name,
    dimension=1024,
    auto_id=True,
    metric_type="COSINE"
)
client.list_collections()


### DML操作

In [ ]:
from langchain_ollama import OllamaEmbeddings
from rich import print as rprint

embed_model = OllamaEmbeddings(model="qwen3-embedding:0.6b")

metadata = client.describe_collection(collection_name=collection_name)
rprint(metadata)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True,
)
chunks = PyPDFLoader("./temp/source_loader_demo/attention.pdf", extraction_mode="layout").load_and_split(
    text_splitter=recursive_splitter)
texts = [chunk.page_content for chunk in chunks]
embeddings = embed_model.embed_documents(texts)
print(len(embeddings))

In [ ]:
# 插入数据
data = [
    {
        "id": i,
        "origin": chunks[i].page_content,
        "vector": embeddings[i]
    } for i in range(len(embeddings))
]

result = client.upsert(collection_name=collection_name, data=data)
rprint(result)

In [ ]:
# 手动落盘
client.flush(collection_name=collection_name)

stas = client.get_collection_stats(collection_name=collection_name)
rprint(stas)

In [ ]:
# 扫描数据
iterator = client.query_iterator(
    collection_name=collection_name,
    filter="",
    output_fields=["id", "origin", "vector"],
)
while True:
    items = iterator.next()
    if not items:
        break
    for item in items:
        print(item["id"])

iterator.close()


# 相似度检索

In [12]:
# 计算 query_vector
query = input("请输入问题：")
query_vector = embed_model.embed_query(query)
client_search = client.search(collection_name=collection_name, data=[query_vector], output_fields=["origin"],
                              limit=10)
rprint(client_search)


data: [[{'id': 469303028512728892, 'distance': 0.7844585180282593, 'entity': {'origin': '4    Why 
Self-Attention'}}, {'id': 469303028512729058, 'distance': 0.7844585180282593, 'entity': {'origin': '4    Why 
Self-Attention'}}, {'id': 469303028512728885, 'distance': 0.6995662450790405, 'entity': {'origin': 'Self-Attention 
(restricted)     O(r·n·d)       O(1)        O(n/r)'}}, {'id': 469303028512729051, 'distance': 0.6995662450790405, 
'entity': {'origin': 'Self-Attention (restricted)     O(r·n·d)       O(1)        O(n/r)'}}, {'id': 
469303028512728858, 'distance': 0.654583215713501, 'entity': {'origin': '3.2    Attention\n\nAn attention function 
can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and 
output are all vectors. The output is computed as a weighted sum\n\n\n                                             
3'}}, {'id': 469303028512729024, 'distance': 0.654583215713501, 'entity': {'origin': '3.2    Attention\n\nAn 
attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query,
keys, values, and output are all vectors. The output is computed as a weighted sum\n\n\n                           
3'}}, {'id': 469303028512728897, 'distance': 0.6504043936729431, 'entity': {'origin': 'computational complexity, 
self-attention layers are faster than recurrent layers when the sequence'}}, {'id': 469303028512729063, 'distance':
0.6504043936729431, 'entity': {'origin': 'computational complexity, self-attention layers are faster than recurrent
layers when the sequence'}}, {'id': 469303028512728867, 'distance': 0.6478981375694275, 'entity': {'origin': 
'queries, keys and values we then perform the attention function in parallel, yieldingdv-dimensional'}}, {'id': 
469303028512729033, 'distance': 0.6478981375694275, 'entity': {'origin': 'queries, keys and values we then perform 
the attention function in parallel, yieldingdv-dimensional'}}]]